# Scenario 1 - Cold Chain Fleet Operations Cockpit

A cold-chain logistics SaaS already ingests telemetry from refrigerated vehicles every 10 seconds. The hard part is not ingestion. The hard part is converting those signals into *live operational decisions*.

**Business problem we are solving**
- Fleet and zone KPIs are often computed by periodic jobs (5-15 minute lag).
- Operators cannot answer "how many vehicles are out of compliance right now?" instantly.
- Slow temperature drift is hard to catch early with threshold-only alerting.

**What this notebook proves with rtbot SQL**
1. Build a 3-layer cockpit (vehicle -> zone -> fleet) using SQL only.
2. Detect excursions and drift warnings incrementally as new data arrives.
3. Query continuously maintained state as DataFrames for operational dashboards.

**Business value**
rtbot adds a real-time analytics layer between existing ingestion and existing dashboards, replacing patchwork cron jobs and custom alert services for these patterns.


In [1]:
import random
import pandas as pd
from rtbot_sql import RtBotSql

random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

import plotly.graph_objects as go


## Stream Definition

Each reading has 8 numeric fields. `door_open` and `compressor_on` are binary flags (0/1). Timestamps are assigned by runtime insertion order.

In [2]:
sql = RtBotSql()

sql.execute('''
CREATE STREAM fleet_telemetry (
    vehicle_id    DOUBLE,
    zone_id       DOUBLE,
    temp_c        DOUBLE,
    setpoint_c    DOUBLE,
    door_open     DOUBLE,
    compressor_on DOUBLE,
    lat           DOUBLE,
    lon           DOUBLE
)
''')


## Deterministic Dataset With Planted Events

We generate 500 readings across 15 vehicles in 3 zones.

- 12 vehicles are healthy (tight temperature around setpoint)
- Vehicle 7 develops instability (drift warning pattern)
- Vehicle 12 has a door-left-open event (rapid excursion)
- Vehicle 14 drifts into excursion later in the run

Target milestones:
- ~100 readings: all healthy
- ~300 readings: drift warning appears
- ~400 readings: first excursion alert appears
- ~500 readings: two active excursions and Zone C underperforming

In [3]:
zones = {
    1: 1, 2: 1, 3: 1, 4: 1, 5: 1,
    6: 2, 7: 2, 8: 2, 9: 2, 10: 2,
    11: 3, 12: 3, 13: 3, 14: 3, 15: 3,
}
healthy = [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 13, 15]

def telemetry_row(vehicle_id: int, seen_count: int):
    zone = zones[vehicle_id]
    setpoint = 2.0
    lat = {1: 34.05, 2: 36.16, 3: 32.78}[zone] + 0.01 * vehicle_id
    lon = {1: -118.24, 2: -115.14, 3: -96.80}[zone] - 0.01 * vehicle_id
    door_open = 0.0
    compressor_on = 1.0

    if vehicle_id in healthy:
        temp_c = 2.0 + random.uniform(-0.30, 0.30)
        if random.random() < 0.02:
            door_open = 1.0
    elif vehicle_id == 7:
        # Stable early, then high short-term volatility.
        if seen_count <= 120:
            temp_c = 2.05 + random.uniform(-0.03, 0.03)
        else:
            temp_c = 2.6 + ((-1) ** seen_count) * random.uniform(0.9, 1.5)
    elif vehicle_id == 14:
        # Slow drift, then late sustained excursion.
        if seen_count <= 30:
            temp_c = 2.2 + random.uniform(-0.08, 0.08)
        elif seen_count <= 50:
            temp_c = 2.8 + random.uniform(-0.2, 0.2)
        else:
            temp_c = 5.1 + random.uniform(-0.35, 0.35)
    elif vehicle_id == 12:
        # Door-open excursion starts after enough history is built.
        if seen_count < 20:
            temp_c = 2.1 + random.uniform(-0.2, 0.2)
        else:
            door_open = 1.0
            compressor_on = 0.0
            temp_c = 6.2 + random.uniform(-0.5, 0.6)
    else:
        temp_c = 2.0 + random.uniform(-0.3, 0.3)

    return [
        float(vehicle_id),
        float(zone),
        float(temp_c),
        float(setpoint),
        float(door_open),
        float(compressor_on),
        float(lat),
        float(lon),
    ]

rows = []
seen = {v: 0 for v in zones}
for i in range(1, 501):
    if i <= 100:
        v = healthy[(i - 1) % len(healthy)]
    elif i <= 300:
        off = i - 101
        if off < 170:
            v = 7
        elif off < 190:
            v = 14
        else:
            v = healthy[off % len(healthy)]
    elif i <= 400:
        off = i - 301
        if off < 65:
            v = healthy[off % len(healthy)]
        else:
            v = 12
    else:
        off = i - 401
        if off < 60:
            v = 14
        elif off < 90:
            v = 12
        else:
            v = healthy[off % len(healthy)]

    seen[v] += 1
    rows.append(telemetry_row(v, seen[v]))

telemetry_df = pd.DataFrame(rows, columns=[
    'vehicle_id', 'zone_id', 'temp_c', 'setpoint_c',
    'door_open', 'compressor_on', 'lat', 'lon'
])

# Plot axis timestamps (10-second cadence) used for didactic visualization.
telemetry_df['event_idx'] = telemetry_df.index + 1
telemetry_df['event_time'] = pd.date_range('2026-01-06 08:00:00', periods=len(telemetry_df), freq='10s')

telemetry_df.head(10)


,vehicle_id,zone_id,temp_c,setpoint_c,door_open,compressor_on,lat,lon,event_idx,event_time
0,1.0,1.0,2.083656,2.0,0.0,1.0,34.06,-118.25,1,2026-01-06 08:00:00
1,2.0,1.0,1.865018,2.0,0.0,1.0,34.07,-118.26,2,2026-01-06 08:00:10
2,3.0,1.0,2.141883,2.0,0.0,1.0,34.08,-118.27,3,2026-01-06 08:00:20
3,4.0,1.0,2.235308,2.0,0.0,1.0,34.09,-118.28,4,2026-01-06 08:00:30
4,5.0,1.0,1.953153,2.0,0.0,1.0,34.10,-118.29,5,2026-01-06 08:00:40
5,6.0,2.0,1.831183,2.0,0.0,1.0,36.22,-115.20,6,2026-01-06 08:00:50
6,8.0,2.0,1.715922,2.0,0.0,1.0,36.24,-115.22,7,2026-01-06 08:01:00
7,9.0,2.0,2.089931,2.0,0.0,1.0,36.25,-115.23,8,2026-01-06 08:01:10
8,10.0,2.0,1.832264,2.0,0.0,1.0,36.26,-115.24,9,2026-01-06 08:01:20
9,11.0,3.0,2.185658,2.0,1.0,1.0,32.89,-96.91,10,2026-01-06 08:01:30


Before creating views, inspect the raw telemetry signal. The traces are shown with line+dots, and we keep gaps in time explicit (no line jump across missing timestamps).

In [4]:
# NOTE: marker_cold_input_plot
axis = telemetry_df[['event_idx', 'event_time']].copy()

fig = go.Figure()
for vid, color in [(1.0, '#4C78A8'), (7.0, '#F58518'), (12.0, '#E45756'), (14.0, '#72B7B2')]:
    d = telemetry_df[telemetry_df['vehicle_id'] == vid][['event_idx', 'temp_c']]
    d = axis.merge(d, on='event_idx', how='left')
    fig.add_trace(
        go.Scatter(
            x=d['event_time'],
            y=d['temp_c'],
            mode='lines+markers',
            connectgaps=False,
            name=f'vehicle {int(vid)}',
            marker=dict(size=4),
            line=dict(width=1.8, color=color),
        )
    )

fig.add_hline(y=2.0, line_dash='dash', line_color='green', annotation_text='setpoint 2.0C')
fig.add_hline(y=4.0, line_dash='dot', line_color='red', annotation_text='excursion threshold 4.0C')
fig.update_layout(
    title='Input telemetry (selected vehicles)',
    xaxis_title='Event time',
    yaxis_title='Temperature (C)',
    template='plotly_white',
    height=520,
)
fig.show()


## Layer 1 - Per-Vehicle Monitoring

`vehicle_status` maintains smoothed temperature, variability, and door-open rate per vehicle key.

In [5]:
sql.execute('''
CREATE MATERIALIZED VIEW vehicle_status AS
  SELECT vehicle_id, zone_id,
         temp_c,
         setpoint_c,
         MOVING_AVERAGE(temp_c, 30)    AS smooth_temp,
         STDDEV(temp_c, 30)            AS temp_variability,
         MOVING_AVERAGE(door_open, 10) AS door_open_rate
  FROM fleet_telemetry
  GROUP BY vehicle_id
''')


## Alert Views

- `excursion_alerts`: smoothed temperature above setpoint + margin
- `drift_warnings`: short volatility rising versus long volatility baseline

In [6]:
sql.execute('''
CREATE MATERIALIZED VIEW excursion_alerts AS
  SELECT vehicle_id, zone_id, temp_c, setpoint_c,
         MOVING_AVERAGE(temp_c, 30) AS smooth_temp
  FROM fleet_telemetry
  GROUP BY vehicle_id
  HAVING MOVING_AVERAGE(temp_c, 30) > setpoint_c + 2.0
''')

sql.execute('''
CREATE MATERIALIZED VIEW drift_warnings AS
  SELECT vehicle_id, zone_id, temp_c,
         STDDEV(temp_c, 30)  AS short_variability,
         STDDEV(temp_c, 120) AS long_variability
  FROM fleet_telemetry
  GROUP BY vehicle_id
  HAVING STDDEV(temp_c, 30) > STDDEV(temp_c, 120) * 1.5
''')


## Layer 2 - Zone Aggregation

Zone-level operational health maintained incrementally.

In [7]:
sql.execute('''
CREATE MATERIALIZED VIEW zone_health AS
  SELECT zone_id,
         COUNT(*) AS active_vehicles,
         AVG(temp_c) AS zone_avg_temp,
         SUM(door_open) AS doors_open_now
  FROM fleet_telemetry
  GROUP BY zone_id
''')


## Layer 3 - Fleet Cockpit + Checkpoint Replay

We replay first N rows at each checkpoint and query the maintained views.

In [8]:
def build_runtime():
    r = RtBotSql()
    r.execute('''
    CREATE STREAM fleet_telemetry (
        vehicle_id DOUBLE,
        zone_id DOUBLE,
        temp_c DOUBLE,
        setpoint_c DOUBLE,
        door_open DOUBLE,
        compressor_on DOUBLE,
        lat DOUBLE,
        lon DOUBLE
    )
    ''')
    r.execute('''
    CREATE MATERIALIZED VIEW vehicle_status AS
      SELECT vehicle_id, zone_id,
             temp_c,
             setpoint_c,
             MOVING_AVERAGE(temp_c, 30)    AS smooth_temp,
             STDDEV(temp_c, 30)            AS temp_variability,
             MOVING_AVERAGE(door_open, 10) AS door_open_rate
      FROM fleet_telemetry
      GROUP BY vehicle_id
    ''')
    r.execute('''
    CREATE MATERIALIZED VIEW excursion_alerts AS
      SELECT vehicle_id, zone_id, temp_c, setpoint_c,
             MOVING_AVERAGE(temp_c, 30) AS smooth_temp
      FROM fleet_telemetry
      GROUP BY vehicle_id
      HAVING MOVING_AVERAGE(temp_c, 30) > setpoint_c + 2.0
    ''')
    r.execute('''
    CREATE MATERIALIZED VIEW drift_warnings AS
      SELECT vehicle_id, zone_id, temp_c,
             STDDEV(temp_c, 30)  AS short_variability,
             STDDEV(temp_c, 120) AS long_variability
      FROM fleet_telemetry
      GROUP BY vehicle_id
      HAVING STDDEV(temp_c, 30) > STDDEV(temp_c, 120) * 1.5
    ''')
    r.execute('''
    CREATE MATERIALIZED VIEW zone_health AS
      SELECT zone_id,
             COUNT(*) AS active_vehicles,
             AVG(temp_c) AS zone_avg_temp,
             SUM(door_open) AS doors_open_now
      FROM fleet_telemetry
      GROUP BY zone_id
    ''')
    return r


def replay_checkpoint(n_rows: int):
    r = build_runtime()
    for _, row in telemetry_df.iloc[:n_rows].iterrows():
        r.execute(
            'INSERT INTO fleet_telemetry VALUES '
            f'({row.vehicle_id}, {row.zone_id}, {row.temp_c}, {row.setpoint_c}, '
            f'{row.door_open}, {row.compressor_on}, {row.lat}, {row.lon})'
        )

    cockpit = r.execute('''
        SELECT COUNT(*) AS total_vehicles,
               AVG(smooth_temp) AS fleet_avg_temp,
               SUM(door_open_rate) AS est_doors_open
        FROM vehicle_status
    ''')
    excursions = r.execute('SELECT * FROM excursion_alerts LIMIT 20')
    drifts = r.execute('SELECT * FROM drift_warnings LIMIT 20')
    zone = r.execute('SELECT * FROM zone_health LIMIT 10')

    print(f'Checkpoint {n_rows}')
    print('- Active excursion vehicles:', sorted(excursions['vehicle_id'].tolist()) if not excursions.empty else [])
    print('- Drift warning vehicles:', sorted(drifts['vehicle_id'].tolist()) if not drifts.empty else [])
    print('- Fleet cockpit (latest):')
    display(cockpit.tail(1) if not cockpit.empty else cockpit)

    if not zone.empty:
        worst_zone = zone.sort_values('zone_avg_temp', ascending=False).head(1)
        print('- Worst-performing zone (highest avg temp):')
        display(worst_zone)

    return {
        'cockpit': cockpit,
        'excursions': excursions,
        'drifts': drifts,
        'zone': zone,
    }

cp100 = replay_checkpoint(100)
cp300 = replay_checkpoint(300)
cp400 = replay_checkpoint(400)
cp500 = replay_checkpoint(500)


Checkpoint 100
- Active excursion vehicles: []
- Drift warning vehicles: []
- Fleet cockpit (latest):


,total_vehicles,fleet_avg_temp,est_doors_open


- Worst-performing zone (highest avg temp):


,zone_id,active_vehicles,zone_avg_temp,doors_open_now
0,1.0,44.0,2.031416,0.0


Checkpoint 300
- Active excursion vehicles: []
- Drift warning vehicles: [7.0]
- Fleet cockpit (latest):


...

,zone_id,active_vehicles,zone_avg_temp,doors_open_now
2,3.0,96.0,2.776081,18.0


Checkpoint 500
- Active excursion vehicles: [12.0, 14.0]
- Drift warning vehicles: [7.0]
- Fleet cockpit (latest):


,total_vehicles,fleet_avg_temp,est_doors_open
227,228.0,2.962585,36.0


- Worst-performing zone (highest avg temp):


,zone_id,active_vehicles,zone_avg_temp,doors_open_now
2,3.0,189.0,3.64271,48.0


## Interactive Inspection Runtime

To make ad-hoc exploration easier, this cell loads the full synthetic dataset into `sql`.
After running it, commands like `sql.execute("SELECT * FROM fleet_telemetry LIMIT 20")` return rows immediately.


In [9]:
# NOTE: marker_cold_interactive_runtime

def load_full_cold_chain_runtime():
    r = build_runtime()
    for _, row in telemetry_df.iterrows():
        r.execute(
            'INSERT INTO fleet_telemetry VALUES '
            f'({row.vehicle_id}, {row.zone_id}, {row.temp_c}, {row.setpoint_c}, '
            f'{row.door_open}, {row.compressor_on}, {row.lat}, {row.lon})'
        )
    return r

sql = load_full_cold_chain_runtime()
print(f'Loaded {len(telemetry_df)} rows into `sql` for interactive inspection.')

# Direct SELECT calls (no helper API needed)
display(sql.execute('SELECT * FROM fleet_telemetry LIMIT 20'))
display(sql.execute('SELECT * FROM vehicle_status LIMIT 20'))
display(sql.execute('SELECT * FROM zone_health LIMIT 20'))
display(sql.execute('SELECT * FROM excursion_alerts LIMIT 20'))
display(sql.execute('SELECT * FROM drift_warnings LIMIT 20'))


Loaded 500 rows into `sql` for interactive inspection.


,vehicle_id,zone_id,temp_c,setpoint_c,door_open,compressor_on,lat,lon
0,12.0,3.0,6.140778,2.0,1.0,0.0,32.90,-96.92
1,12.0,3.0,5.781490,2.0,1.0,0.0,32.90,-96.92
2,12.0,3.0,6.392390,2.0,1.0,0.0,32.90,-96.92
3,12.0,3.0,5.758970,2.0,1.0,0.0,32.90,-96.92
4,12.0,3.0,5.864117,2.0,1.0,0.0,32.90,-96.92
5,12.0,3.0,6.319124,2.0,1.0,0.0,32.90,-96.92
6,12.0,3.0,6.034219,2.0,1.0,0.0,32.90,-96.92
7,12.0,3.0,6.793310,2.0,1.0,0.0,32.90,-96.92
8,12.0,3.0,5.830297,2.0,1.0,0.0,32.90,-96.92
9,12.0,3.0,6.540888,2.0,1.0,0.0,32.90,-96.92


,vehicle_id,zone_id,temp_c,setpoint_c,smooth_temp,temp_variability,door_open_rate
0,7.0,2.0,4.055345,2.0,2.556727,1.238969,0.0
1,12.0,3.0,6.540888,2.0,6.206881,0.338253,1.0
2,14.0,3.0,4.885882,2.0,5.052762,0.172342,0.0


,zone_id,active_vehicles,zone_avg_temp,doors_open_now
0,1.0,83.0,1.998706,1.0
1,2.0,228.0,2.153510,1.0
2,3.0,189.0,3.642710,48.0


,vehicle_id,zone_id,temp_c,setpoint_c,smooth_temp
0,12.0,3.0,6.540888,2.0,6.206881
1,14.0,3.0,4.885882,2.0,5.052762


,vehicle_id,zone_id,temp_c,short_variability,long_variability
0,7.0,2.0,1.547166,1.235353,0.820823


Now overlay alert outputs on top of the same input signal: orange circles are drift warnings and red X markers are excursion alerts.

In [10]:
# NOTE: marker_cold_detection_plot
def build_time_index(runtime_sql, source_stream: str):
    msgs = runtime_sql.get_store().read(source_stream)
    return {msg.timestamp: i + 1 for i, msg in enumerate(msgs)}


def view_history_df(runtime_sql, view_name: str, source_stream: str, source_axis: pd.DataFrame):
    meta = runtime_sql.get_catalog().lookup_view(view_name)
    cols = [name for name, _ in sorted(meta.field_map.items(), key=lambda kv: kv[1])]
    msgs = runtime_sql.get_store().read(view_name)
    if not msgs:
        return pd.DataFrame(columns=['event_idx', 'event_time'] + cols)

    time_index = build_time_index(runtime_sql, source_stream)
    rows = []
    for msg in msgs:
        row = {col: msg.values[i] for i, col in enumerate(cols)}
        row['event_idx'] = time_index.get(msg.timestamp)
        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.dropna(subset=['event_idx']).copy()
    df['event_idx'] = df['event_idx'].astype(int)
    df = df.merge(source_axis, on='event_idx', how='left')
    return df

source_axis = telemetry_df[['event_idx', 'event_time']].copy()
exc_df = view_history_df(sql, 'excursion_alerts', 'fleet_telemetry', source_axis)
drift_df = view_history_df(sql, 'drift_warnings', 'fleet_telemetry', source_axis)

fig = go.Figure()
for vid, color in [(7.0, '#F58518'), (12.0, '#E45756'), (14.0, '#72B7B2')]:
    d = telemetry_df[telemetry_df['vehicle_id'] == vid][['event_idx', 'temp_c']]
    d = source_axis.merge(d, on='event_idx', how='left')
    fig.add_trace(
        go.Scatter(
            x=d['event_time'],
            y=d['temp_c'],
            mode='lines+markers',
            connectgaps=False,
            name=f'input vehicle {int(vid)}',
            marker=dict(size=3),
            line=dict(width=1.5, color=color),
            opacity=0.75,
        )
    )

if not drift_df.empty:
    fig.add_trace(
        go.Scatter(
            x=drift_df['event_time'],
            y=drift_df['temp_c'],
            mode='markers',
            name='drift warning',
            marker=dict(size=11, color='#FF9D00', symbol='circle-open', line=dict(width=2)),
        )
    )

if not exc_df.empty:
    fig.add_trace(
        go.Scatter(
            x=exc_df['event_time'],
            y=exc_df['temp_c'],
            mode='markers',
            name='excursion alert',
            marker=dict(size=12, color='#D62728', symbol='x'),
        )
    )

fig.add_hline(y=2.0, line_dash='dash', line_color='green', annotation_text='setpoint 2.0C')
fig.add_hline(y=4.0, line_dash='dot', line_color='red', annotation_text='excursion threshold 4.0C')
fig.update_layout(
    title='Alert overlays on telemetry',
    xaxis_title='Event time',
    yaxis_title='Temperature (C)',
    template='plotly_white',
    height=560,
)
fig.show()


## Business Case

What this replaces:
- Periodic batch KPI jobs for fleet/zone rollups
- Separate service logic for excursion/drift alerting
- Custom backend logic for dashboard-ready operational state

What this enables:
- Real-time per-vehicle and zone-level visibility using SQL only
- Faster intervention via rolling-statistical early warnings
- Sub-millisecond cockpit reads over maintained incremental state

Integration model: rtbot sits between existing ingestion and existing dashboards, adding the real-time analytics layer without replacing either.